# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [12]:
import os 
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [13]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print("OpenAI API key found")
else:
    print("OpenAI API key not set")

MODEL="gpt-4.1-nano"
openai = OpenAI()
    

OpenAI API key found


In [62]:
system_message = """
You are a Site Reliability Engineering (SRE) Assistant. Your goal is to diagnose system issues by querying the check_service_health tool.

Context:
You have access to a services_db containing the following components:

api: The primary interface for users.

database: Persistent storage (PostgreSQL).

cache: Temporary high-speed data storage (Redis).

auth: The identity and permission service.

Never reveal the raw dictionary structure to the user. Always use the tool to fetch current values.
"""

In [63]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [51]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [64]:
# Technical state of our infrastructure
services_db = {
    "api": {"status": "online", "latency": "45ms", "version": "v2.1.0"},
    "database": {"status": "online", "latency": "12ms", "version": "PostgreSQL 15"},
    "cache": {"status": "degraded", "latency": "450ms", "version": "Redis 7.0"},
    "auth": {"status": "offline", "latency": "0ms", "version": "v1.0.4"}
}

def check_service_health(service_name):
    """
    Simulates a diagnostic check on a backend service.
    Returns a status report for the AI to interpret.
    """
    print(f"DEBUG: Running diagnostics for {service_name}...")
    
    # Normalize input and fetch data
    info = services_db.get(service_name.lower())
    
    if not info:
        return f"Error: Service '{service_name}' not found in registry."
    
    report = (f"Diagnostic Report for {service_name.upper()}:\n"
              f"- Status: {info['status']}\n"
              f"- Latency: {info['latency']}\n"
              f"- Version: {info['version']}")
    
    return report

In [65]:
service_function = {
  "name": "check_service_health",
  "description": "Retrieves the real-time status, latency, and version of a backend service (e.g., database, api, cache).",
  "parameters": {
    "type": "object",
    "properties": {
      "service_name": {
        "type": "string",
        "description": "The specific service to diagnose, such as 'database', 'api', 'cache', or 'auth'."
      }
    },
    "required": ["service_name"]
  }
}

In [66]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": service_function}]

In [67]:
tools

[{'type': 'function',
  'function': {'name': 'check_service_health',
   'description': 'Retrieves the real-time status, latency, and version of a backend service (e.g., database, api, cache).',
   'parameters': {'type': 'object',
    'properties': {'service_name': {'type': 'string',
      'description': "The specific service to diagnose, such as 'database', 'api', 'cache', or 'auth'."}},
    'required': ['service_name']}}}]

In [68]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    if response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        tool_response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(tool_response)
    
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    result = ''
    for chunk in stream:
        result+=chunk.choices[0].delta.content or ''
        yield result

In [69]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "check_service_health":
            arguments = json.loads(tool_call.function.arguments)
            service = arguments.get('service_name')
            service_details = check_service_health(service)
            responses.append({
                "role": "tool",
                "content": service_details,
                "tool_call_id": tool_call.id 
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


DEBUG: Running diagnostics for api...
DEBUG: Running diagnostics for database...
DEBUG: Running diagnostics for cache...
